In [ ]:
from pathlib import Path
import sys
PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

from src.utils.utils import load_file
from src.practice2.rag.document_paser import parse_manual
from src.practice2.rag.text_chunker import chunk_documents
from src.practice2.rag.embeddings import vectorize_chunks
from src.practice2.rag.retriever import retrieve
from src.practice2.rag.generator import generate_answer

manual_path = PROJECT_DIR / "data" / "manual.pdf"

chunk_size = 1000
chunk_overlap = 150
top_k = 4

mineru_mode = "flash"
embed_model = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

gen_model = "gpt-4.1-mini"
api_key = load_file(".env")

c:\Users\jia\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
docs = parse_manual(manual_path, mode=mineru_mode)

Project root: C:\Users\jia\Downloads\llm\LLM_Practice2
Output folder: C:\Users\jia\Downloads\llm\LLM_Practice2\src\output

[1/5] Parsing with MinerU (mode=flash)...
- parsed documents/pages: 1
- saved: C:\Users\jia\Downloads\llm\LLM_Practice2\src\output\parsed.md


In [7]:
chunks = chunk_documents(
    docs,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content)


[2/5] Chunking...
- chunks: 40
- saved: C:\Users\jia\Downloads\llm\LLM_Practice2\src\output\chunks.jsonl
Number of chunks: 40
## 사용 설명서

## Music Studio 7 (HW-LS7\*H) Music Studio 5 (HW-LS5\*H)

보다 완벽한 서비스를 받으시려면 구입하신 제품을 www.samsung.com에서등록하시기 바랍니다.사용 설명서 상의 그림은 설명을 위해 그려진 것으로 실제제품 외관과 다를 수 있습니다.

제품 디자인 및 제품 규격은 제품 성능 향상을 위해 사전 예고 없이 변경될 수있습니다

## 목차


In [8]:
encoder, vectors = vectorize_chunks(
    chunks,
    embed_model,
)

print("Embedding shape:", vectors.shape)


[3/5] Vectorizing with sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.28s/it]

- embedding shape: (40, 384)
- saved: C:\Users\jia\Downloads\llm\LLM_Practice2\src\output\embeddings.npy
Embedding shape: (40, 384)


In [9]:
query = "What is camera calibration?"

retrieved = retrieve(
    query,
    encoder,
    vectors,
    chunks,
    top_k=top_k,
)


[4/5] Retrieving relevant chunks...
#1 score=0.281, page=?: rowspan=1 colspan=1><img src="images/507bfc587a39ae9eff676d8452d8d9a0c28f2ff44613644011ab00023423684a.jpg"/></td><td rowspan=1 colspan=1>주의,...
#2 score=0.269, page=?: <table><tr><td colspan="1" rowspan="1">음향모드</td><td colspan="1" rowspan="1">•음악:음악 콘텐츠를 위해 더 자연스럽고 공간을 가득채우는 듯한 사운드를 제공합니다.표준:오리지널 사운드로 감상할 ...
#3 score=0.257, page=?: ·가능하면 코어리스 HDMI케이블을 사용하는 것이 좋습니다.코어형 HDMI케이블을 사용하는 경우에는 직경이 14mm미만인 것을 사용하세요.  ·HDMI케이블이 ARC를 지원하지 않는 경우에는 이 기능을 사용할 수없습니다.  방송 신호가 Dolby일 때...
#4 score=0.247, page=?: 재생</td><td colspan="1" rowspan="1">감상 중이었던 음악을 바로 재생 할 수 있습니다.("즉시 재생 기능 사용하기"색션을 참조하세요.)</td></tr><tr><td>음향 세부항목 설정</td><td>Voice 음향 강화는 동...


In [ ]:
answer = generate_answer(
    query,
    retrieved,
    gen_model,
    api_key,
)

print(answer)

### Agent

In [ ]:
import json
from src.practice2.agent.speaker_agent import create_speaker_agent, speaker_state

In [ ]:
agent = create_speaker_agent(
    retriever=retrieved,
)

print("Speaker Agent is ready!")

Speaker Agent is ready!


In [ ]:
def show_speaker_state():
    print("\nCurrent Mock Speaker State:")
    print(
        json.dumps(
            speaker_state,
            ensure_ascii=False,
            indent=2,
        )
    )


def run_agent(question):
    print("\n" + "=" * 60)
    print("USER:", question)
    print("=" * 60)

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    })

    print("\nAGENT ANSWER:")
    print(result["messages"][-1].content)

    show_speaker_state()

    print("\nTOOL CALLS:")
    for message in result["messages"]:
        if getattr(message, "tool_calls", None):
            for tool_call in message.tool_calls:
                print(
                    "Tool:",
                    tool_call.get("name"),
                    "| Arguments:",
                    tool_call.get("args"),
                )

    return result

In [ ]:
run_agent("현재 스피커 상태를 확인해 줘.")
run_agent("음량을 35로 변경해 줘.")
run_agent("야간 모드를 켜 줘.")
run_agent(
    "블루투스 연결이 잘 안 돼. "
    "매뉴얼을 확인하고 해결 방법을 알려줘."
)
run_agent(
    "현재 블루투스 연결을 해제하고 "
    "페어링 모드로 바꿔 줘."
)

In [ ]:
from copy import deepcopy
from src.agent.speaker_state import DEFAULT_SPEAKER_STATE

speaker_state.clear()
speaker_state.update(deepcopy(DEFAULT_SPEAKER_STATE))

show_speaker_state()